# In Class Activity April 14th, 2026

In [ ]:
# !pip install optuna
# !pip install sweetviz


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]



### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [3]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [4]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





The data seems to be well cleaned and without missing values we would need to remove/fill. The response variable, Income, appears to be somewhat imbalanced with over 75% of the rows falling into the <= 50k class. This will need to be mitigated so models will train well on the data.

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True,
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}')


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The model so far is alright, ~0.7 F1 scores after CV with no noticable good or bad splits. Tuning and class balancing will likely improve the results of future models.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building.
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'scale_pos_weight': [1, 5, 10],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'subsample': [0.6, 0.8, 1.0]
}

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [9]:
grid_search = GridSearchCV(XGBClassifier(enable_categorical=True, random_state=42),
                           param_grid, cv=skf, scoring='f1')
grid_search.fit(X, y)
print(f'Best params: {grid_search.best_params_}')
print(f'Best F1: {grid_search.best_score_:.4f}')

Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'scale_pos_weight': 1, 'subsample': 1.0}
Best F1: 0.7151


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [10]:
# tuning xgboost classifier with RandomizedSearchCV
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    'n_estimators': np.arange(100, 350, step=50),
    'subsample': np.linspace(0.6, 1.0, num=5)
}

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')

xgb_random_best = xgb_random.best_estimator_
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')

Best parameters from RandomizedSearchCV: {'subsample': np.float64(0.7), 'scale_pos_weight': np.float64(3.0), 'n_estimators': np.int64(150), 'max_depth': np.int64(8), 'learning_rate': np.float64(0.07444444444444444)}
Best F1 score from RandomizedSearchCV: 0.7161481388324681
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7431
           1       0.63      0.84      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.79      0.84      0.81      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [11]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 300, step=50)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)

    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight,
                               max_depth=max_depth, learning_rate=learning_rate,
                               n_estimators=n_estimators, subsample=subsample,
                               enable_categorical=True)

    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

xgb_optuna_best = XGBClassifier(random_state=42, enable_categorical=True, **study.best_params)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')

[I 2026-04-15 10:42:53,419] A new study created in memory with name: no-name-dd177908-83a1-4942-b9f9-c94b98f2df55


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 10:42:57,742] Trial 0 finished with value: 0.7039417881951556 and parameters: {'scale_pos_weight': 6.4678771676345885, 'max_depth': 8, 'learning_rate': 0.24315893990984905, 'n_estimators': 300, 'subsample': 0.9650303868429001}. Best is trial 0 with value: 0.7039417881951556.
[I 2026-04-15 10:43:00,287] Trial 1 finished with value: 0.6671215251672357 and parameters: {'scale_pos_weight': 9.60597960091081, 'max_depth': 4, 'learning_rate': 0.2559033352383362, 'n_estimators': 300, 'subsample': 0.6638008117819144}. Best is trial 0 with value: 0.7039417881951556.
[I 2026-04-15 10:43:03,971] Trial 2 finished with value: 0.7180187096462811 and parameters: {'scale_pos_weight': 2.8443179306841095, 'max_depth': 9, 'learning_rate': 0.12133859174102543, 'n_estimators': 200, 'subsample': 0.8576048822029055}. Best is trial 2 with value: 0.7180187096462811.
[I 2026-04-15 10:43:05,151] Trial 3 finished with value: 0.6602460925912428 and parameters: {'scale_pos_weight': 7.386948059598547, '

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


I probably prefer the Optuna method, It took around 2 minutes to run vs the 1 min for random and 27 for grid, and it provided narrowly better results than random search. Both Optuna and RS had ~0.85 weighted avg F1 scores with ~0.9 for class 0 and a much worse 0.72 for class 1. Overall, with this data there are probably other steps we can take to improve the model but for tuning purposes I prefer Optuna.